# 02_build_artifacts: CPU dựng CAA, classifier K-Steer/eval và ORBIT global plane

Input: RUN_ROOT hoàn thành 01, có cả build và activation. Không load backbone, không load judge, không lọc cặp. Đọc CPU shards đã lưu để dựng artifact. Training classifier có optimizer và minibatch cursor để resume. Không extract activation lần nữa.

Mỗi notebook chạy từ kernel mới. Thay PREVIOUS_RUN_INPUT bằng đường dẫn Add Input thật; stage 00 đặt None. Khi tiếp tục cùng notebook, dùng output checkpoint mới nhất của chính notebook đó.

Chưa có kết quả T4 được chạy sẵn trong notebook này.

In [ ]:
# Giữ các tham số giống nhau trong cả sáu notebook của workflow paper mới.
MODE = "smoke"  # smoke -> pilot -> full; POS/NEG 96 tokens, không judge lọc
TARGET_MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
SPLIT_MODE = "question_holdout"  # hoặc shared_pool; không đổi giữa các stage
PRECISION_PROFILE = "fp16"  # fp16 | judge_nf4 | low_memory_nf4; run riêng cho mỗi profile
POS_NEG_SOURCE = "backbone"  # workflow inline chỉ dùng backbone để thu activation đúng model
RUN_LABEL = "tonebank-paper-smoke-v2"  # run mới; không dùng checkpoint workflow cũ
BUNDLE_ZIP = None  # Tự tìm ZIP hoặc bundle đã giải nén; cũng có thể đặt đường dẫn thư mục
PREVIOUS_RUN_INPUT = "/kaggle/input/OUTPUT-NOTEBOOK-TRUOC/orbit_runs/tonebank-paper-smoke-v2/meta-llama--Llama-3.2-3B-Instruct"
TIME_BUDGET_SECONDS = 30000  # tổng budget notebook, gồm setup; chừa thời gian Save Version


In [ ]:
"""Standard-library bootstrap, embedded in every notebook before package import."""
import hashlib
import json
import shutil
import sys
import zipfile
from pathlib import Path


def _bundle_file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def _is_bundle_root(path):
    return all((path / name).is_file() for name in (
        "orbit_kaggle/__init__.py", "orbit_kaggle/session.py", "configs/smoke.json",
        "requirements/runtime.lock.txt", "vendor/source_manifest.json"))


def _bundle_roots(path):
    if _is_bundle_root(path):
        return [path]
    candidates = {p.parent.parent for p in path.rglob("source_manifest.json")
                  if p.parent.name == "vendor" and _is_bundle_root(p.parent.parent)}
    # Kaggle can recursively expand archived releases inside the outer bundle.
    return sorted(p for p in candidates if not any(parent in candidates for parent in p.parents))


def locate_bundle(bundle_path=None, input_root="/kaggle/input"):
    """BUNDLE_ZIP remains compatible; its value can now also be a directory."""
    root = Path(bundle_path) if bundle_path is not None else Path(input_root)
    if not root.exists():
        raise FileNotFoundError(f"Bundle input không tồn tại: {root}")
    if root.is_file():
        if not zipfile.is_zipfile(root):
            raise ValueError(f"Bundle file không phải ZIP: {root}")
        return root
    if bundle_path is not None and _is_bundle_root(root):
        return root
    zips = sorted(root.rglob("orbit_kaggle_bundle.zip"))
    if len(zips) == 1:
        return zips[0]
    roots = _bundle_roots(root)
    if not zips and len(roots) == 1:
        return roots[0]
    raise ValueError(
        "Cần đúng một bundle. BUNDLE_ZIP có thể là đường dẫn ZIP HOẶC thư mục "
        "chứa orbit_kaggle/, configs/, vendor/, requirements/. "
        f"ZIP tìm thấy: {[str(p) for p in zips]}; thư mục bundle: {[str(p) for p in roots]}")


def _verify_bundle_tree(root):
    manifest_path = root / "bundle_files.sha256.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("schema") != 1 or not isinstance(manifest.get("files"), dict):
            raise ValueError("Sai schema bundle_files.sha256.json")
        for name, checksum in manifest["files"].items():
            path = root / name
            if not path.resolve().is_relative_to(root.resolve()):
                raise ValueError(f"Unsafe manifest path: {name}")
            if not path.is_file() or _bundle_file_hash(path) != checksum:
                raise ValueError(f"Bundle checksum mismatch: {name}")
        # Every executable/config file must be covered, including after auto-extraction.
        actual = {str(p.relative_to(root)) for p in root.rglob("*") if p.is_file()
                  and p.name not in ("bundle_files.sha256.json", "orbit_kaggle_bundle.sha256")
                  and "__pycache__" not in p.parts}
        if actual != set(manifest["files"]):
            raise ValueError("Bundle file list differs from checksum manifest")
        return manifest["files"]
    # Compatibility with the first released ZIP when Kaggle has removed the ZIP.
    # Its original outer ZIP hash cannot verify the expanded adapter files.
    source = json.loads((root / "vendor/source_manifest.json").read_text())
    for name, checksum in source["sha256"].items():
        if name.startswith(("angular_", "ksteering/src/steering/", "ksteering/src/utils/")) or name in ("ksteering/src/__init__.py", "ksteering/LICENSE"):
            path = root / "vendor" / name
            if not path.is_file() or _bundle_file_hash(path) != checksum:
                raise ValueError(f"Legacy vendor checksum mismatch: {name}")
    if _bundle_file_hash(root / "vendor/source_prompts.py") != source["generated_prompt_sha256"]:
        raise ValueError("Legacy prompt checksum mismatch")
    print("Bundle cũ đã giải nén: đã kiểm checksum vendor; chưa có checksum đầy đủ "
          "cho adapter/config. Bundle mới có bundle_files.sha256.json để kiểm toàn bộ file.")
    return {str(p.relative_to(root)): _bundle_file_hash(p) for p in root.rglob("*")
            if p.is_file() and "__pycache__" not in p.parts}


def prepare_bundle(bundle_path=None, input_root="/kaggle/input", destination_base="/tmp/orbit-code"):
    import tempfile
    source = locate_bundle(bundle_path, input_root)
    with tempfile.TemporaryDirectory(prefix="orbit-bootstrap-") as tmp:
        if source.is_file():
            checksum_file = source.with_suffix(".sha256")
            if not checksum_file.is_file():
                raise FileNotFoundError("Đính kèm orbit_kaggle_bundle.sha256 bên cạnh ZIP")
            if _bundle_file_hash(source) != checksum_file.read_text().split()[0]:
                raise ValueError("Sai checksum bundle ZIP")
            tree = Path(tmp)
            with zipfile.ZipFile(source) as archive:
                for member in archive.infolist():
                    if not (tree / member.filename).resolve().is_relative_to(tree.resolve()):
                        raise ValueError("Unsafe ZIP path")
                archive.extractall(tree)
        else:
            tree = source
        if not _is_bundle_root(tree):
            raise ValueError(f"Thiếu module/config trong bundle: {tree}")
        files = _verify_bundle_tree(tree)
        fingerprint = hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest()
        destination = Path(destination_base) / fingerprint
        if not destination.exists():
            destination.mkdir(parents=True)
            for name in files:
                target = destination / name
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(tree / name, target)
            if (tree / "bundle_files.sha256.json").exists():
                shutil.copy2(tree / "bundle_files.sha256.json", destination / "bundle_files.sha256.json")
        for name, checksum in files.items():
            if not (destination / name).is_file() or _bundle_file_hash(destination / name) != checksum:
                raise ValueError(f"Local code cache checksum mismatch: {name}")
        loaded = sys.modules.get("orbit_kaggle")
        if loaded is not None and Path(loaded.__file__).resolve().parent != destination / "orbit_kaggle":
            raise RuntimeError("Bundle đã đổi trong kernel đang dùng. Restart kernel rồi chạy lại.")
    print("Code input:", source, "→ CODE_ROOT:", destination)
    return destination

"""Pinned worker Python independent of the notebook's pip/venv/ensurepip.

Embedded in notebooks as well as shipped in the bundle, so the updated notebook
can also launch the first release's already attached code Dataset.
"""
import hashlib
import io
import json
import os
import platform
import subprocess
import sys
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

WORKER_PYTHON = "3.12.11"
UV_VERSION = "0.8.22"
PIP_VERSION = "25.2"
UV_WHEEL_URL = "https://files.pythonhosted.org/packages/3a/a9/a83cee9b8cf63e57ce64ba27c77777cc66410e144fd178368f55af1fa18d/uv-0.8.22-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
UV_WHEEL_SHA256 = "8efec4ef5acddc35f0867998c44e0b15fc4dace1e4c26d01443871a2fbb04bf6"


def _runtime_write_json(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, sort_keys=True, indent=2) + "\n")
    tmp.replace(path)


def _runtime_run(command, env):
    print("Runtime setup:", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), env=env, check=True)


def ensure_uv(tools_root):
    """Extract a checksum-pinned native wheel; requires neither host pip nor root."""
    if sys.platform != "linux" or platform.machine() not in ("x86_64", "AMD64"):
        raise RuntimeError("This Kaggle runtime bootstrap supports Linux x86_64")
    directory = Path(tools_root) / ("uv-" + UV_VERSION)
    directory.mkdir(parents=True, exist_ok=True)
    wheel = directory / "uv.whl"
    if not wheel.exists():
        print("Downloading checksum-pinned uv", UV_VERSION, flush=True)
        try:
            with urllib.request.urlopen(UV_WHEEL_URL, timeout=120) as response:
                data = response.read()
        except (OSError, urllib.error.URLError) as exc:
            raise RuntimeError("Không tải được uv bootstrap. Bật Internet trên Kaggle và thử lại; không cần apt/sudo/ensurepip.") from exc
        if hashlib.sha256(data).hexdigest() != UV_WHEEL_SHA256:
            raise ValueError("uv wheel checksum mismatch")
        temporary = wheel.with_suffix(".tmp")
        temporary.write_bytes(data)
        temporary.replace(wheel)
    data = wheel.read_bytes()
    if hashlib.sha256(data).hexdigest() != UV_WHEEL_SHA256:
        raise ValueError("Cached uv wheel checksum mismatch")
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        members = [name for name in archive.namelist() if Path(name).name == "uv"]
        if len(members) != 1:
            raise ValueError("Unexpected uv wheel executable layout")
        executable = archive.read(members[0])
    uv = directory / "uv"
    if uv.exists() and uv.read_bytes() != executable:
        raise ValueError("Cached uv executable checksum mismatch")
    if not uv.exists():
        temporary = uv.with_suffix(".tmp")
        temporary.write_bytes(executable)
        temporary.chmod(0o755)
        temporary.replace(uv)
    return uv


def runtime_environment(tools_root):
    """Keep managed Python, cache and executable links inside the task directory."""
    root = Path(tools_root)
    env = dict(os.environ)
    env.update(UV_PYTHON_INSTALL_DIR=str(root / "python"), UV_PYTHON_BIN_DIR=str(root / "bin"),
               UV_CACHE_DIR=str(root / "cache"), UV_NATIVE_TLS="true")
    # Worker environments must not inherit another interpreter's site-packages.
    for name in ("PYTHONHOME", "PYTHONPATH", "VIRTUAL_ENV", "CONDA_PREFIX"):
        env.pop(name, None)
    return env


def create_worker_environment(env_root, tools_root):
    env_root, tools_root = Path(env_root), Path(tools_root)
    uv = ensure_uv(tools_root)
    env = runtime_environment(tools_root)
    python = env_root / "bin/python"
    if not python.exists():
        _runtime_run([uv, "--no-config", "python", "install", WORKER_PYTHON], env)
        _runtime_run([uv, "--no-config", "venv", "--managed-python", "--python", WORKER_PYTHON, env_root], env)
    probe = json.loads(subprocess.check_output(
        [str(python), "-I", "-c", "import sys,json; print(json.dumps({'version':list(sys.version_info[:3]),'executable':sys.executable}))"],
        env=env, text=True))
    if probe["version"] != [int(x) for x in WORKER_PYTHON.split(".")]:
        raise RuntimeError(f"Unexpected worker Python {probe}; expected {WORKER_PYTHON}. Use a fresh environment path.")
    return python, uv, env, probe


def install_env(code_root, env_root="/tmp/orbit-runtime", cpu=False, tools_root=None):
    lock = Path(code_root) / "requirements/runtime.lock.txt"
    identity = {"schema": 2, "lock_sha256": hashlib.sha256(lock.read_bytes()).hexdigest(), "cpu": bool(cpu),
                "worker_python": WORKER_PYTHON, "uv": UV_VERSION, "uv_wheel_sha256": UV_WHEEL_SHA256,
                "pip": PIP_VERSION, "torch": "2.6.0", "torchvision": "0.21.0", "bitsandbytes": None if cpu else "0.48.2"}
    fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    base = Path(env_root)
    # A failed legacy /tmp/orbit-runtime created by Python 3.13 is left untouched.
    actual = base.with_name(base.name + "-py312-" + fingerprint[:16])
    tools_root = Path(tools_root) if tools_root else base.parent / "orbit-tools"
    marker = actual / "orbit-install.json"
    if marker.exists():
        saved = json.loads(marker.read_text())
        if saved.get("fingerprint") != fingerprint:
            raise RuntimeError("Environment fingerprint mismatch")
        python = actual / "bin/python"
        if not python.exists():
            raise RuntimeError("Installed environment lost its Python executable")
        return python
    actual.parent.mkdir(parents=True, exist_ok=True)
    python, uv, env, probe = create_worker_environment(actual, tools_root)
    state = actual / "orbit-install-state.json"
    _runtime_write_json(state, {"identity": identity, "status": "running", "worker": probe})
    try:
        # uv seeds pip without invoking the notebook interpreter's ensurepip.
        _runtime_run([uv, "--no-config", "pip", "install", "--python", python, f"pip=={PIP_VERSION}"], env)
        index = "https://download.pytorch.org/whl/cpu" if cpu else "https://download.pytorch.org/whl/cu124"
        _runtime_run([python, "-m", "pip", "install", "--only-binary=:all:", "torch==2.6.0", "torchvision==0.21.0", "--index-url", index], env)
        _runtime_run([python, "-m", "pip", "install", "--only-binary=:all:", "-r", lock], env)
        if not cpu:
            _runtime_run([python, "-m", "pip", "install", "--only-binary=:all:", "bitsandbytes==0.48.2"], env)
        _runtime_run([python, "-m", "pip", "check"], env)
    except BaseException as exc:
        _runtime_write_json(state, {"identity": identity, "status": "failed", "worker": probe,
                                    "error": f"{type(exc).__name__}: {exc}"})
        raise
    result = {"identity": identity, "fingerprint": fingerprint, "worker": probe,
              "host_python": sys.version, "status": "completed"}
    _runtime_write_json(marker, result)
    _runtime_write_json(state, result)
    print("Worker ready:", python, "Python", WORKER_PYTHON, flush=True)
    return python

"""Standard-library checkpoint validation and small download exports.

Embedded in the notebooks so it also works with an attached legacy code bundle.
"""
import hashlib
import json
import shutil
import zipfile
from pathlib import Path


def validate_checkpoint(root):
    root = Path(root)
    index = root / "checkpoint_index.json"
    info = json.loads(index.read_text())
    if info.get("schema") != 1 or not isinstance(info.get("archives"), list) or not info["archives"]:
        raise ValueError(f"Invalid checkpoint index: {index}")
    seen = set()
    for part in info["archives"]:
        name = part["name"]
        if not isinstance(name, str) or Path(name).name != name or name in seen:
            raise ValueError(f"Unsafe or duplicate checkpoint part: {name}")
        seen.add(name)
        path = root / name
        if not path.is_file():
            raise FileNotFoundError(f"Missing checkpoint part in current session: {path}")
        if path.stat().st_size != part["bytes"]:
            raise ValueError(f"Checkpoint size mismatch: {path}")
        checksum = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(2**20), b""):
                checksum.update(block)
        if checksum.hexdigest() != part["sha256"]:
            raise ValueError(f"Checkpoint checksum mismatch: {path}")
    return info


def export_checkpoint_zip(root, destination, max_bytes=64 * 1024**2):
    """Export a verified small checkpoint. Return None above the explicit size cap.

    ZIP contains the index and original gzip parts directly at its root, so a
    Kaggle auto-expanded Dataset can be passed straight to restore_run().
    This only creates a local file; it does not save a Kaggle version.
    """
    root, destination = Path(root), Path(destination)
    info = validate_checkpoint(root)
    files = [root / "checkpoint_index.json"] + [root / part["name"] for part in info["archives"]]
    total = sum(path.stat().st_size for path in files)
    if total > max_bytes:
        return None
    destination.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(destination.parent).free < total + 1024**2:
        raise OSError(f"Not enough free disk space for checkpoint download ZIP: {destination}")
    temporary = destination.with_name(destination.name + ".tmp")
    try:
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
            for path in files:
                archive.write(path, path.name)
        with zipfile.ZipFile(temporary) as archive:
            if archive.testzip() is not None:
                raise ValueError("Checkpoint download ZIP failed CRC validation")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination

import os, sys, json, hashlib, shutil, zipfile, subprocess, time
from pathlib import Path
NOTEBOOK_START = time.monotonic()
assert TIME_BUDGET_SECONDS > 0
assert sys.version_info >= (3, 11), "Notebook bootstrap cần Python >=3.11; worker sẽ dùng Python 3.12 riêng"
CODE_ROOT = prepare_bundle(BUNDLE_ZIP)
assert (CODE_ROOT / "orbit_kaggle/fused_build.py").is_file(), "Upload Code Dataset bằng bundle paper mới; bundle cũ không có fused build"
sys.path.insert(0, str(CODE_ROOT))
os.environ["PYTHONPATH"] = str(CODE_ROOT)
os.environ["HF_HOME"] = "/tmp/orbit-hf"  # weight cache không bị Save Output vô tình
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from orbit_kaggle.session import restore_run, run_stage
PYTHON = install_env(CODE_ROOT)
# HF_TOKEN chỉ ở environment; không in token, không ghi token vào artifact.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    if TARGET_MODEL_ID.startswith("meta-llama/"):
        raise RuntimeError("Bật secret HF_TOKEN có quyền đọc model Llama gated") from exc
config = json.loads((CODE_ROOT / "configs" / f"{MODE}.json").read_text())
config["target_model_id"] = TARGET_MODEL_ID
config["split"]["mode"] = SPLIT_MODE
profile = json.loads((CODE_ROOT / "configs/precision_profiles.json").read_text())[PRECISION_PROFILE]
config["precision"] = profile["precision"]
config["judge"]["precision"] = profile["judge_precision"]
config["build"]["response_source"] = POS_NEG_SOURCE
assert config["build"].get("extraction") == "generation_prefill", "Cần config workflow paper mới"
RUN_ROOT = Path("/kaggle/working/orbit_runs") / RUN_LABEL / TARGET_MODEL_ID.replace("/", "--")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
if PREVIOUS_RUN_INPUT:
    restore_run(PREVIOUS_RUN_INPUT, RUN_ROOT)
if (RUN_ROOT / "checkpoint_index.json").exists():
    restore_run(RUN_ROOT, RUN_ROOT)  # re-open own compacted checkpoint in this same session
(RUN_ROOT / "runtime_environment.json").write_text((Path(PYTHON).parents[1] / "orbit-install.json").read_text())
CONFIG_PATH = RUN_ROOT / "resolved_config.json"
if CONFIG_PATH.exists():
    assert json.loads(CONFIG_PATH.read_text()) == config, "Config khác checkpoint; dùng RUN_LABEL mới hoặc giữ đúng config cũ"
else:
    CONFIG_PATH.write_text(json.dumps(config, indent=2) + "\n")
CHECKPOINTED = False
def stage(name, **kwargs):
    global CHECKPOINTED
    if CHECKPOINTED:
        print("SKIPPED until resume:", name)
        return 75
    intentional = kwargs.pop("allow_checkpoint", False)
    remaining = int(TIME_BUDGET_SECONDS - (time.monotonic() - NOTEBOOK_START))
    if remaining <= 0:
        CHECKPOINTED = True
        print("Notebook time budget reached. Snapshot current run, save output and resume.")
        return 75
    code = run_stage(PYTHON, CONFIG_PATH, RUN_ROOT, name, remaining, allow_checkpoint=True, **kwargs)
    if code == 75 and not intentional:
        CHECKPOINTED = True
        print("CHECKPOINT SAVED ON SESSION DISK. Save this version's output and resume this notebook; upstream stage incomplete.")
    return code
print("Run:", RUN_ROOT, "model:", TARGET_MODEL_ID, "mode:", MODE, "split:", SPLIT_MODE, "precision:", profile)


In [ ]:
stage("artifacts")

In [ ]:
# Đóng gói checkpoint trước Save Version để tránh hàng chục nghìn output files.
# Archive theo phần <=1GiB uncompressed; bản unpacked còn giữ trong /tmp của phiên.
from orbit_kaggle.session import snapshot_run
snapshot = snapshot_run(RUN_ROOT)
validate_checkpoint(RUN_ROOT)  # cũng kiểm được bundle cũ; không chỉ tin log hoặc index
print(json.dumps(snapshot, indent=2))
checkpoint_files = [RUN_ROOT / "checkpoint_index.json"] + [RUN_ROOT / part["name"] for part in snapshot["archives"]]
for path in checkpoint_files:
    assert path.is_file(), f"Thiếu checkpoint trên đĩa phiên: {path}"
    print("SESSION FILE:", path, "bytes:", path.stat().st_size)
download_zip = export_checkpoint_zip(RUN_ROOT, Path("/kaggle/working") / f"orbit-checkpoint-{RUN_ROOT.parent.name}-{RUN_ROOT.name}.zip")
if download_zip is not None:
    print("DOWNLOAD ZIP (ngay trong /kaggle/working):", download_zip, "bytes:", download_zip.stat().st_size)
else:
    print("Checkpoint >64MiB: không tạo ZIP bản sao; giữ index và archive parts trong RUN_ROOT.")
print("Các file trên mới tồn tại trong phiên. Chỉ tiếp tục khi saved version có Output files thực tế; log cell không đủ.")

## Giữ output và tiếp tục

Nếu stage báo checkpoint/time budget: không chạy stage tiếp theo. Dùng **Save Version** ở góc trên phải; chọn **Quick Save → Advanced Settings → Save output files** nếu giao diện có lựa chọn đó. Chờ version lưu xong, mở đúng version → **Output** và xác nhận có `orbit_runs/.../checkpoint_index.json` cùng các `checkpoint-part-*.tar.gz`.

**Nếu Output ghi 0 files:** version đó chưa có checkpoint để notebook sau đọc, dù vẫn hiện log cell cũ. Bấm Edit, chạy lại riêng cell snapshot khi phiên vẫn còn RUN_ROOT và kiểm tra các dòng SESSION FILE. Lưu version có output files. Nếu phiên đã mất file, chạy lại notebook từ các input đã lưu. Có thể chọn **Save & Run All**, nhưng thao tác này chạy một phiên sạch từ đầu, không chuyển file tương tác sang; khi resume phải khai báo PREVIOUS_RUN_INPUT đã được lưu thật. README có cách tải checkpoint dự phòng nếu phiên còn file.

Ở notebook tiếp theo chọn **Add Input → Notebook Outputs**, tìm notebook/version đã có file, Add. Copy đường dẫn RUN_ROOT dưới `/kaggle/input/...` vào PREVIOUS_RUN_INPUT. Chỉ đính kèm một snapshot cumulative mới nhất của run. Không coi file còn trong `/kaggle/working` là đã bền vững. Xem README_KAGGLE.md cho cách giới hạn output và lặp qua ba backbone.